# OSC variant vcf scoring

In [1]:
%%bash
module load miniconda3/24.1.2-py310
module load cuda/12.4.1
conda activate py311

In [1]:
from alphagenome_research.model import dna_model
from alphagenome import colab_utils
from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.interpretation import ism
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
# os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.9'
# os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# os.environ["TF_GPU_ALLOCATOR"]='cuda_malloc_async'

import matplotlib.pyplot as plt
import pandas as pd
import pysam
from pysam import VariantFile
from io import StringIO
from tqdm import tqdm
import os
# from dotenv import load_dotenv

pd.set_option('display.max_columns', None)


In [3]:
LMNA_START = 156_082_572
LMNA_END = 156_140_081
gene_symbol = "LMNA"
LMNA_INTERVAL = genome.Interval('chr1', 156_082_572, 156_140_081)

BASE_PATH = '/users/PAS2905/coraalbers/'

AG_DATA_PATH = '/users/PAS2905/coraalbers/ag/ag_data/'

HG38_FASTA_PATH = '/users/PAS2905/coraalbers/ag/hg38.fa'
HG38_GTF_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.gtf.gz.feather'
HG38_SPLICE_START_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.splice_sites_starts.feather'
HG38_SPLICE_END_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.splice_sites_ends.feather'

CLINVAR_PATH = '/users/PAS2905/coraalbers/ag/clinvar.vcf.gz'

HEART_UB = 'UBERON:0000948'
LV_UB = 'UBERON:0002084'

gtf = pd.read_feather(
    HG38_GTF_PATH
)

In [4]:
model = dna_model.create_from_huggingface( 
    'all_folds', 
    organism_settings={ 
        dna_model.Organism.HOMO_SAPIENS: dna_model.OrganismSettings( 
            fasta_path=HG38_FASTA_PATH, 
            
            gtf_feather_path=HG38_GTF_PATH, 
            splice_site_starts_feather_path=HG38_SPLICE_START_PATH, 
            splice_site_ends_feather_path=HG38_SPLICE_END_PATH, 
        ), dna_model.Organism.MUS_MUSCULUS: dna_model.OrganismSettings() } )

print('all folds model initialized!')

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

all folds model initialized!


In [19]:
gene_symbol = "LMNA"
window_bp = 1_000_000

gene_interval = gene_annotation.get_gene_interval(gtf, gene_symbol=gene_symbol)
region = gene_interval.pad(window_bp, window_bp)
vcf_contig = region.chromosome.removeprefix("chr")

vcf_path = CLINVAR_PATH
with VariantFile(vcf_path) as vcf_in:
    nearby = [
        rec for rec in vcf_in.fetch(
            vcf_contig,
            max(region.start - 1, 0),
            region.end,
        )
    ]
print(f"{len(nearby)} variants within ±{window_bp:,} bp of {gene_symbol}")
print(f"Region: {region.chromosome}:{region.start}-{region.end}")

13076 variants within ±1,000,000 bp of LMNA
Region: chr1:155082571-157140081


In [20]:
def variants_near_gene(vcf_path, gene_symbol, gtf, window_bp=1_000_000):
    region = gene_interval.pad(window_bp, window_bp)
    contig = to_vcf_contig(region.chromosome)
    gene_interval = gene_annotation.get_gene_interval(gtf, gene_symbol=gene_symbol)
    with VariantFile(vcf_path) as vcf_in:
        # nearby = [
        #     rec for rec in vcf_in.fetch(
        #         vcf_contig,
        #         max(region.start - 1, 0),
        #         region.end,
        #     )
        # ]
        # print(f"{len(nearby)} variants within ±{window_bp:,} bp of {gene_symbol}")
        # print(f"Region: {region.chromosome}:{region.start}-{region.end}")

        for record in vcf_in.fetch(contig, max(region.start - 1, 0), region.end):
            yield record
        
    

In [21]:
variants_near_gene(vcf_path, 'LMNA', gtf, 1_000_000)

<generator object variants_near_gene at 0x148014a442a0>

In [22]:
def to_vcf_contig(chrom: str) -> str:
    return chrom.removeprefix("chr")  # 'chr1' -> '1'

def variants_near_gene(vcf_in, gene_interval, window_bp=1_000_000):
    region = gene_interval.pad(window_bp, window_bp)
    contig = to_vcf_contig(region.chromosome)
    for record in vcf_in.fetch(contig, max(region.start - 1, 0), region.end):
        yield record

# # write a filtered vcf file with variants near the gene
# with VariantFile(vcf_path) as vcf_in, VariantFile(f"{BASE_PATH}ag/variant-effects/osc/outputs/lmna_within_1Mb.vcf", "w", header=vcf_in.header) as vcf_out:
#     for record in variants_near_gene(vcf_in, gene_interval):
#         vcf_out.write(record)

In [23]:
PATHOGENIC = {
    "Pathogenic",
    "Likely_pathogenic",
    "Pathogenic/Likely_pathogenic",
    "Uncertain_significance"
    
}

def is_pathogenic(record, gene_filter=False):
    clnsig = record.info.get("CLNSIG")
    if clnsig is None:
        return False
    # pysam returns a tuple for Number=. fields
    if gene_filter:
        if "LMNA:" in record.info.get("GENEINFO", ""):
            if isinstance(clnsig, tuple):
                return any(sig in PATHOGENIC for sig in clnsig)

    else: 
        if isinstance(clnsig, tuple):
            return any(sig in PATHOGENIC for sig in clnsig)
        
    return clnsig in PATHOGENIC




In [46]:
nearby_vcf = f'{BASE_PATH}ag/variant-effects/osc/outputs/lmna_within_1Mb.vcf'

# with VariantFile(nearby_vcf) as nearby_in, \
#      VariantFile(f'{BASE_PATH}ag/variant-effects/osc/outputs/lmna_variants_pathogenic_VUS_LMNA.vcf', "w", header=nearby_in.header) as vcf_out:
#     count = 0
#     for record in nearby_in.fetch():
#         if is_pathogenic(record, gene_filter=True): 
#             # print(record)
#             vcf_out.write(record)
#             count += 1
# print(f"Wrote {count} pathogenic variants")

## convert vcf to csv

In [16]:
def get_vcf_names(vcf_path):
    """Finds the true column header line inside the VCF file."""
    with open(vcf_path, "r") as f:
        for line in f:
            if line.startswith("#CHROM"):
                # Remove the leading '#' and split by tabs
                return line.strip("#").strip().split("\t")
    raise ValueError("No header line starting with '#CHROM' found.")


vcf_filename = f'{BASE_PATH}ag/variant-effects/osc/outputs/plp_with_nc.bed'
csv_filename = f'{BASE_PATH}ag/variant-effects/osc/outputs/plp_with_nc.csv'

column_names = ['CHROM', 'POS', 'ID', 'REF','ALT', 'QUAL', 'FILTER', 'INFO', 'NC_CHROM', 'NC_START', 'NC_END', 'BP_OVERLAP']

# Read the VCF file, skipping metadata rows starting with '##'
df = pd.read_csv(
    vcf_filename,
    comment="#",
    sep="\t",
    names=column_names,
    header=None,
    low_memory=False,
    index_col=False
)



In [17]:
df

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,NC_CHROM,NC_START,NC_END,BP_OVERLAP
0,1,156114912,66757,GCCGGCCATGGAGACC,G,.,.,ALLELEID=77654;CLNDISDB=MedGen:CN380145|MedGen...,1,156081998,156114918,7
1,1,156114913,2690636,CCGGCCATGGAGACCCCGTCCCAG,C,.,.,ALLELEID=2851669;CLNDISDB=MedGen:C3661900;CLND...,1,156081998,156114918,6
2,1,156115275,200933,G,A,.,.,ALLELEID=196457;CLNDISDB=MedGen:CN230736|MedGe...,1,156115274,156126203,1
3,1,156115275,228271,G,C,.,.,ALLELEID=228273;CLNDISDB=MedGen:C3661900|EFO:E...,1,156115274,156126203,1
4,1,156115276,2705963,T,A,.,.,"ALLELEID=2877288;CLNDISDB=MONDO:MONDO:0018993,...",1,156115274,156126203,1
...,...,...,...,...,...,...,...,...,...,...,...,...
61,1,156137651,66856,C,G,.,.,ALLELEID=77753;CLNDISDB=MedGen:CN380145|MedGen...,1,156137232,156137653,1
62,1,156137652,2018990,A,G,.,.,"ALLELEID=2069226;CLNDISDB=MONDO:MONDO:0018993,...",1,156137232,156137653,1
63,1,156137653,179809,G,A,.,.,"ALLELEID=172349;CLNDISDB=MONDO:MONDO:0018993,M...",1,156137232,156137653,1
64,1,156138486,1285503,A,G,.,.,"ALLELEID=1275346;CLNDISDB=MONDO:MONDO:0007906,...",1,156137761,156138487,1


In [18]:
df['CHROM'] = 'chr1'
df

# Export the clean structure to a CSV
df.to_csv(csv_filename, index=False)
print(f"Successfully exported genomic VCF data to {csv_filename}!")

Successfully exported genomic VCF data to /users/PAS2905/coraalbers/ag/variant-effects/osc/outputs/plp_with_nc.csv!


## predict variants
from https://www.alphagenomedocs.com/colabs/batch_variant_scoring.html 

In [20]:
csv_file = f'{BASE_PATH}ag/variant-effects/osc/outputs/plp_with_nc.csv'
vcf = pd.read_csv(csv_file, sep=',')
print(vcf.columns)
required_columns = ['ID', 'CHROM', 'POS', 'REF', 'ALT']
for column in required_columns:
  if column not in vcf.columns:
    raise ValueError(f'VCF file is missing required column: {column}.')

organism = 'human'  # @param ["human", "mouse"] {type:"string"}

# @markdown Specify length of sequence around variants to predict:
sequence_length = '1MB'  # @param ["16KB", "100KB", "500KB", "1MB"] { type:"string" }
sequence_length = dna_client.SUPPORTED_SEQUENCE_LENGTHS[
    f'SEQUENCE_LENGTH_{sequence_length}'
]


# @markdown Specify which scorers to use to score your variants:
score_rna_seq = True  # @param { type: "boolean"}
score_cage = True  # @param { type: "boolean" }
score_procap = True  # @param { type: "boolean" }
score_atac = True  # @param { type: "boolean" }
score_dnase = True  # @param { type: "boolean" }
score_chip_histone = True  # @param { type: "boolean" }
score_chip_tf = True  # @param { type: "boolean" }
score_polyadenylation = False  # @param { type: "boolean" }
score_splice_sites = True  # @param { type: "boolean" }
score_splice_site_usage = True  # @param { type: "boolean" }
score_splice_junctions = True  # @param { type: "boolean" }



# Parse organism specification.
organism_map = {
    'human': dna_client.Organism.HOMO_SAPIENS,
    'mouse': dna_client.Organism.MUS_MUSCULUS,
}
organism = organism_map[organism]

# Parse scorer specification.
scorer_selections = {
    'rna_seq': score_rna_seq,
    'cage': score_cage,
    'procap': score_procap,
    'atac': score_atac,
    'dnase': score_dnase,
    'chip_histone': score_chip_histone,
    'chip_tf': score_chip_tf,
    'polyadenylation': score_polyadenylation,
    'splice_sites': score_splice_sites,
    'splice_site_usage': score_splice_site_usage,
    'splice_junctions': score_splice_junctions,
}

all_scorers = variant_scorers.RECOMMENDED_VARIANT_SCORERS
# print(all_scorers)
selected_scorers = [
    all_scorers[key]
    for key in all_scorers
    if scorer_selections.get(key.lower(), False)
]



# Remove any scorers or output types that are not supported for the chosen organism.
unsupported_scorers = [
    scorer
    for scorer in selected_scorers
    if (
        organism.value
        not in variant_scorers.SUPPORTED_ORGANISMS[scorer.base_variant_scorer]
    )
    | (
        (scorer.requested_output == dna_client.OutputType.PROCAP)
        & (organism == dna_client.Organism.MUS_MUSCULUS)
    )
]
if len(unsupported_scorers) > 0:
  print(
      f'Excluding {unsupported_scorers} scorers as they are not supported for'
      f' {organism}.'
  )
  for unsupported_scorer in unsupported_scorers:
    selected_scorers.remove(unsupported_scorer)





Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO',
       'NC_CHROM', 'NC_START', 'NC_END', 'BP_OVERLAP'],
      dtype='object')


In [21]:
# Score variants in the VCF file.
results = []

for i, vcf_row in tqdm(vcf.iterrows(), total=len(vcf)):
    variant = genome.Variant(
      chromosome=str(vcf_row.CHROM),
      position=int(vcf_row.POS),
      reference_bases=vcf_row.REF,
      alternate_bases=vcf_row.ALT,
      name=vcf_row.ID,
    )
    
    interval = LMNA_INTERVAL.resize(sequence_length)
    
    variant_scores = model.score_variant(
      interval=interval,
      variant=variant,
      variant_scorers=selected_scorers,
      organism=organism,
    )
    results.append(variant_scores)

df_scores = variant_scorers.tidy_scores(results)


# @markdown Other settings:
download_predictions = True  # @param { type: "boolean" }

if download_predictions:
  df_scores.to_csv(f'{BASE_PATH}ag/variant-effects/osc/outputs/plp_with_nc.{sequence_length}.scores.csv', index=False)
#   files.download('variant_scores.csv')

print('completed')

  0%|          | 0/66 [00:00<?, ?it/s]E0803 13:12:35.848855 1049314 gpu_hlo_schedule.cc:968] The byte size of input/output arguments (33217052848) exceeds the base limit (31805407232). This indicates an error in the calculation!
W0803 13:12:35.850225 1049314 hlo_rematerialization.cc:3231] Can't reduce memory use below 20.62GiB (22144696496 bytes) by rematerialization; only reduced to 20.62GiB (22144696672 bytes), down from 20.62GiB (22144696672 bytes) originally
W0803 13:12:45.891346 1049314 bfc_allocator.cc:514] Allocator (GPU_0_bfc) ran out of memory trying to allocate 2.13GiB (rounded to 2290089984)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
W0803 13:12:45.891938 1049314 bfc_allocator.cc:525] *******************************************___*************************_*****____*****____******____
  0%|          

JaxRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 2.13GiB. [tf-allocator-allocation-error=''] [executable_name='jit__apply_aggregation']

In [35]:
# Examine just the effects of the variants on specific ontology term
columns = [c for c in df_scores.columns if c != 'ontology_curie']
df_scores[(df_scores['ontology_curie'] == LV_UB)][columns]

,variant_id,scored_interval,gene_id,gene_name,gene_type,gene_strand,junction_Start,junction_End,output_type,variant_scorer,track_name,track_strand,Assay title,biosample_name,biosample_type,biosample_life_stage,data_source,endedness,genetically_modified,transcription_factor,histone_mark,gtex_tissue,raw_score
387,chr1:155085095:C>T,chr1:154822951-155347239:.,None,None,None,None,None,None,DNASE,"CenterMaskScorer(requested_output=DNASE, width...",UBERON:0000948 DNase-seq,.,DNase-seq,heart,tissue,embryonic,encode,paired,False,NaN,NaN,NaN,0.008914
2883,chr1:155085095:C>T,chr1:154822951-155347239:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0000948 Histone ChIP-seq H3K27me3,.,Histone ChIP-seq,heart,tissue,embryonic,encode,single,False,NaN,H3K27me3,NaN,0.011138
2884,chr1:155085095:C>T,chr1:154822951-155347239:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0000948 Histone ChIP-seq H3K4me1,.,Histone ChIP-seq,heart,tissue,embryonic,encode,single,False,NaN,H3K4me1,NaN,-0.005752
2885,chr1:155085095:C>T,chr1:154822951-155347239:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0000948 Histone ChIP-seq H3K4me3,.,Histone ChIP-seq,heart,tissue,embryonic,encode,single,False,NaN,H3K4me3,NaN,0.014744
2886,chr1:155085095:C>T,chr1:154822951-155347239:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0000948 Histone ChIP-seq H3K9ac,.,Histone ChIP-seq,heart,tissue,embryonic,encode,single,False,NaN,H3K9ac,NaN,0.005914
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23608,chr1:155085095:C>T,chr1:154822951-155347239:.,ENSG00000143590,EFNA3,protein_coding,+,None,None,SPLICE_SITE_USAGE,GeneMaskSplicingScorer(requested_output=SPLICE...,usage_UBERON:0000948 total RNA-seq,+,total RNA-seq,heart,tissue,adult,encode,NaN,NaN,NaN,NaN,,0.015625
24339,chr1:155085095:C>T,chr1:154822951-155347239:.,ENSG00000251246,EFNA4-EFNA3,protein_coding,+,155079069,155085315,SPLICE_JUNCTIONS,SpliceJunctionScorer(),junction_UBERON:0000948 polyA plus RNA-seq,.,polyA plus RNA-seq,heart,tissue,adult,encode,NaN,NaN,NaN,NaN,,0.151367
24340,chr1:155085095:C>T,chr1:154822951-155347239:.,ENSG00000143590,EFNA3,protein_coding,+,155079069,155085315,SPLICE_JUNCTIONS,SpliceJunctionScorer(),junction_UBERON:0000948 polyA plus RNA-seq,.,polyA plus RNA-seq,heart,tissue,adult,encode,NaN,NaN,NaN,NaN,,0.151367
24341,chr1:155085095:C>T,chr1:154822951-155347239:.,ENSG00000251246,EFNA4-EFNA3,protein_coding,+,155079069,155085315,SPLICE_JUNCTIONS,SpliceJunctionScorer(),junction_UBERON:0000948 total RNA-seq,.,total RNA-seq,heart,tissue,adult,encode,NaN,NaN,NaN,NaN,,0.149902


In [42]:
lmna_var_scores = pd.read_csv(f'{BASE_PATH}ag/variant-effects/osc/outputs/lmna_variants_pathogenic_VUS_LMNA.524288.scores.csv')
lmna_var_scores

,variant_id,scored_interval,gene_id,gene_name,gene_type,gene_strand,junction_Start,junction_End,output_type,variant_scorer,track_name,track_strand,Assay title,ontology_curie,biosample_name,biosample_type,biosample_life_stage,data_source,endedness,genetically_modified,transcription_factor,histone_mark,gtex_tissue,raw_score
0,chr1:156114693:C>T,chr1:155852549-156376837:.,NaN,NaN,NaN,NaN,NaN,NaN,ATAC,"CenterMaskScorer(requested_output=ATAC, width=...",CL:0000084 ATAC-seq,.,ATAC-seq,CL:0000084,T-cell,primary_cell,adult,encode,paired,False,NaN,NaN,NaN,-0.185082
1,chr1:156114693:C>T,chr1:155852549-156376837:.,NaN,NaN,NaN,NaN,NaN,NaN,ATAC,"CenterMaskScorer(requested_output=ATAC, width=...",CL:0000100 ATAC-seq,.,ATAC-seq,CL:0000100,motor neuron,in_vitro_differentiated_cells,adult,encode,paired,False,NaN,NaN,NaN,0.135565
2,chr1:156114693:C>T,chr1:155852549-156376837:.,NaN,NaN,NaN,NaN,NaN,NaN,ATAC,"CenterMaskScorer(requested_output=ATAC, width=...",CL:0000236 ATAC-seq,.,ATAC-seq,CL:0000236,B cell,primary_cell,adult,encode,paired,False,NaN,NaN,NaN,-0.195181
3,chr1:156114693:C>T,chr1:155852549-156376837:.,NaN,NaN,NaN,NaN,NaN,NaN,ATAC,"CenterMaskScorer(requested_output=ATAC, width=...",CL:0000623 ATAC-seq,.,ATAC-seq,CL:0000623,natural killer cell,primary_cell,adult,encode,paired,False,NaN,NaN,NaN,-0.185215
4,chr1:156114693:C>T,chr1:155852549-156376837:.,NaN,NaN,NaN,NaN,NaN,NaN,ATAC,"CenterMaskScorer(requested_output=ATAC, width=...",CL:0000624 ATAC-seq,.,ATAC-seq,CL:0000624,"CD4-positive, alpha-beta T cell",primary_cell,adult,encode,paired,False,NaN,NaN,NaN,-0.186056
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29702,chr1:156114693:C>T,chr1:155852549-156376837:.,ENSG00000160789,LMNA,protein_coding,+,156134528.0,156135900.0,SPLICE_JUNCTIONS,SpliceJunctionScorer(),junction_UBERON:0036149 total RNA-seq,.,total RNA-seq,UBERON:0036149,suprapubic skin,tissue,adult,encode,NaN,NaN,NaN,NaN,NaN,0.354980
29703,chr1:156114693:C>T,chr1:155852549-156376837:.,ENSG00000160789,LMNA,protein_coding,+,156137028.0,156139079.0,SPLICE_JUNCTIONS,SpliceJunctionScorer(),junction_UBERON:0036149 total RNA-seq,.,total RNA-seq,UBERON:0036149,suprapubic skin,tissue,adult,encode,NaN,NaN,NaN,NaN,NaN,0.297119
29704,chr1:156114693:C>T,chr1:155852549-156376837:.,ENSG00000160789,LMNA,protein_coding,+,156137743.0,156139079.0,SPLICE_JUNCTIONS,SpliceJunctionScorer(),junction_UBERON:0036149 total RNA-seq,.,total RNA-seq,UBERON:0036149,suprapubic skin,tissue,adult,encode,NaN,NaN,NaN,NaN,NaN,0.237793
29705,chr1:156114693:C>T,chr1:155852549-156376837:.,ENSG00000160789,LMNA,protein_coding,+,156130773.0,156137112.0,SPLICE_JUNCTIONS,SpliceJunctionScorer(),junction_UBERON:0036149 total RNA-seq,.,total RNA-seq,UBERON:0036149,suprapubic skin,tissue,adult,encode,NaN,NaN,NaN,NaN,NaN,0.205200


In [43]:
lmna_var_scores['variant_id'].unique()

array(['chr1:156114693:C>T'], dtype=object)

In [ ]:
# # Score variants in the VCF file.
# results = []

# for i, vcf_row in tqdm(vcf.iterrows(), total=len(vcf)):
#   variant = genome.Variant(
#       chromosome=str(vcf_row.CHROM),
#       position=int(vcf_row.POS),
#       reference_bases=vcf_row.REF,
#       alternate_bases=vcf_row.ALT,
#       name=vcf_row.ID,
#   )
#   interval = variant.reference_interval.resize(sequence_length)

#   variant_scores = model.score_variant(
#       interval=interval,
#       variant=variant,
#       variant_scorers=selected_scorers,
#       organism=organism,
#   )
#   results.append(variant_scores)

# df_scores = variant_scorers.tidy_scores(results)


# # @markdown Other settings:
# download_predictions = True  # @param { type: "boolean" }

# if download_predictions:
#   df_scores.to_csv(f'{BASE_PATH}ag/variant-effects/osc/outputs/lmna_variants_pathogenic_VUS_LMNA.scores.csv', index=False)
# #   files.download('variant_scores.csv')

# df_scores

In [ ]:
# df_scores.to_pickle('lmna_pathogenic_variant_scores.pkl')
# # @markdown Other settings:
# download_predictions = True  # @param { type: "boolean" }

# if download_predictions:
#   df_scores.to_csv('/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_variant_scores.csv', index=False)
#   # files.download('/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_variant_scores.csv')

In [ ]:
columns = [c for c in df_scores.columns if c != 'ontology_curie']
heart_df_scores = df_scores[(df_scores['ontology_curie'] == 'UBERON:0000948')][columns] # heart uberon identifier
heart_df_scores.to_csv('/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_variant_scores_heart_UBERON0000948.csv', index=False)

In [18]:
heart_df_scores.head()

,variant_id,scored_interval,gene_id,gene_name,gene_type,gene_strand,junction_Start,junction_End,output_type,variant_scorer,track_name,track_strand,Assay title,biosample_name,biosample_type,biosample_life_stage,data_source,endedness,genetically_modified,transcription_factor,histone_mark,gtex_tissue,raw_score,quantile_score
387,chr1:156114693:C>T,chr1:156049157-156180229:.,None,None,None,None,None,None,DNASE,"CenterMaskScorer(requested_output=DNASE, width...",UBERON:0000948 DNase-seq,.,DNase-seq,heart,tissue,embryonic,encode,paired,False,NaN,NaN,NaN,0.141282,0.963955
2883,chr1:156114693:C>T,chr1:156049157-156180229:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0000948 Histone ChIP-seq H3K27me3,.,Histone ChIP-seq,heart,tissue,embryonic,encode,single,False,NaN,H3K27me3,NaN,-0.177345,-0.999621
2884,chr1:156114693:C>T,chr1:156049157-156180229:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0000948 Histone ChIP-seq H3K4me1,.,Histone ChIP-seq,heart,tissue,embryonic,encode,single,False,NaN,H3K4me1,NaN,-0.000861,-0.134700
2885,chr1:156114693:C>T,chr1:156049157-156180229:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0000948 Histone ChIP-seq H3K4me3,.,Histone ChIP-seq,heart,tissue,embryonic,encode,single,False,NaN,H3K4me3,NaN,0.088124,0.988467
2886,chr1:156114693:C>T,chr1:156049157-156180229:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0000948 Histone ChIP-seq H3K9ac,.,Histone ChIP-seq,heart,tissue,embryonic,encode,single,False,NaN,H3K9ac,NaN,0.101405,0.989576
